# Tarea 2 – Parte 1: Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Configuración | Valor |
| :--- | :--- |
| Fecha de extracción | **2026-10-23** (hora de Lima) |
| Corpus A – documentos | 25 imágenes de documentos peruanos: 8 impresos nítidos, 8 fotos de celular y 9 manuscritos |
| Corpus B – audio | 23 clips (48,9 min) en español: 2 audios de YouTube, 9 entrevistas propias y 12 frases de OpenSLR 73 |
| OCR | **PaddleOCR 3.7.0** (PaddleX 3.7.2): detector `PP-OCRv6_medium_det` + reconocedor `PP-OCRv6_medium_rec`, `lang="es"`, motor PyTorch (`engine="transformers"`) en CPU |
| ASR | **faster-whisper 1.2.1** (CTranslate2 4.6.0) con **Whisper large-v3-turbo** en int8, CPU, `beam_size=5`, `language="es"` |
| VAD | Silero VAD de faster-whisper (`vad_filter=True`, pausas de 300 ms) |
| Diarización | Huellas de voz WeSpeaker ResNet34 (`pyannote.audio` 4.0.7) + clustering jerárquico por coseno, adaptado de `voz.py` (Sesión 13) |
| Seudonimización | Candidatos con spaCy `es_core_news_md` 3.8.0 (NER) y expresiones regulares, revisión manual y reemplazo difuso con `seudonimizar` de `voz.py` |

**Fuentes y licencias** (detalle por archivo en `datos/fuentes.csv`):

| Archivos | Fuente | Licencia |
| :--- | :--- | :--- |
| `impreso1`–`impreso8` | Págs. 2, 4, 9, 15, 16, 22, 32 y 36 de las bases integradas de la AS N° 013-2024-INDECOPI-1, buscador público del SEACE | Texto oficial sin derecho de autor (D. Leg. 822, art. 9 b) |
| `foto1`–`foto8` | Fotos con celular, tomadas por la estudiante, de la Directiva Ejecutiva Regional N° 002-2024-GR/GOB.REG.TACNA | Texto oficial sin derecho de autor (D. Leg. 822, art. 9 b) |
| `manuscrito3`, `6`, `8`, `9` | Wikimedia Commons (Academia Nacional de la Historia de Argentina y Biblioteca Nacional del Perú) | Dominio público o CC0 |
| `manuscrito1`, `2`, `4`, `5`, `7` | Notas de la BNP, El Peruano, BYU y Revista de Arte | Texto en dominio público; fotografía con derechos reservados: **uso académico local, sin redistribución** |
| 2 audios `.mp4` | Grabación propia de videos de YouTube, según la estudiante | Licencia del video por confirmar: **uso académico local, sin redistribución** |
| 9 audios `.ogg` / `.aac` | Entrevistas breves grabadas por la estudiante | Grabación propia con consentimiento informado, según la estudiante; el audio no se publica |
| 12 clips `.wav` | OpenSLR 73, *Crowdsourced high-quality Peruvian Spanish speech data set* (Google) | CC BY-SA 4.0; Guevara-Rukoz et al. (2020), LREC |

**Entorno.** Las versiones exactas están en `requirements-notebooks.txt`. En esta laptop (Windows 11, solo CPU, 7,3 GB de RAM) se fijaron `torch==2.8.0`, `ctranslate2==4.6.0` y `onnxruntime==1.20.1`, porque las versiones más nuevas no cargan con el runtime de Visual C++ instalado (`WinError 1114`). Por el mismo motivo PaddleOCR corre con el motor PyTorch: con PaddlePaddle 3.3.1 y oneDNN el detector falla en CPU.

**Privacidad.** Este notebook se publica en GitHub con sus salidas. Por eso solo muestra textos ya seudonimizados y conteos. La tabla de nombres reales y sus seudónimos, los textos crudos y los audios se quedan en `datos/raw/`, que no se versiona.

In [1]:
import gc
import hashlib
import io
import json
import os
import random
import re
import time
import urllib.request
import urllib.robotparser
import warnings
import zipfile
from datetime import date
from importlib.metadata import version
from pathlib import Path

# Antes de importar los modelos: sin chequeo de servidores de PaddleX ni barras de progreso
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import truststore
from PIL import Image

# Valida HTTPS con los certificados del sistema operativo. En Windows, el antivirus
# intercepta HTTPS con su propio certificado raíz, que el certifi de requests no conoce.
truststore.inject_into_ssl()

# Plotly: interactivo en Jupyter/VS Code + copia PNG para que GitHub muestre los gráficos
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 110)

USER_AGENT = ("Mozilla/5.0 (compatible; TareaDiplomadoPython/1.0; uso academico; "
              "+https://github.com/IvettexMamani/diplomado-python-2026)")

DATOS = Path("datos")
RAW = DATOS / "raw"
DOCS_RAW = RAW / "documentos"
AUDIO_RAW = RAW / "audio"
FECHA = date.today().isoformat()

PAQUETES = ["paddleocr", "paddlex", "torch", "faster-whisper", "ctranslate2", "onnxruntime",
            "pyannote.audio", "spacy", "es_core_news_md", "rapidfuzz", "plotly", "pandas"]
print("Fecha de ejecución:", FECHA)
pd.Series({p: version(p) for p in PAQUETES}, name="versión").to_frame()

Fecha de ejecución: 2026-10-23


,versión
paddleocr,3.7.0
paddlex,3.7.2
torch,2.8.0
faster-whisper,1.2.1
ctranslate2,4.6.0
onnxruntime,1.20.1
pyannote.audio,4.0.7
spacy,3.8.16
es_core_news_md,3.8.0
rapidfuzz,3.14.6
